## はじめに
ここでは因子グラフから確率分布を計算するアルゴリズムである確率伝搬法(belief propagation)についてまとめる。確率伝搬法はSum-productメッセージ伝達法 (sum-product message passing)とも呼ばれる。

レーティングアルゴリズムのTrueSkillで利用されているようで、TrueSkillの数理に関する資料を読んでもさっぱりだったので、基礎的な部分を簡単な例ををもとに理解を深める

## 因子グラフと確率伝搬法について

変数ノード$\nu_{k}$における周辺分布の値は、そのノードに流れ込んでくるすべてのメッセージの積を計算すればよい。$F_{\nu_{k}}$はその変数につながる因子集合で、周辺分布は入ってくるメッセージの積で表される。$Z$は正規化定数。
$$
p({\nu_{k}}) = \frac{1}{Z}\prod_{f \in F_{\nu_{k}}} m_{f \to \nu_{k}}(\nu_{k})
$$

因子ノードから変数ノードへのメッセージは、計算したい変数$\nu_{j}$をのぞくすべての変数について、因子関数の値と他のメッセージの積を積分(和)したものである。そして、因子$f$に送るときは、$f$以外の因子から来た情報を全部掛けることで自分の情報のダブルカウントを避ける。

$$
m_{f \to \nu_{j}}(\nu_{j}) = \int ... \int f(\mathbf{V}) \prod_{i \neq j}m_{\nu_{i} \to f}(\nu_{i}) d\mathbf{v_{i}}
$$

変数ノード$\nu_{k}$から因子$f$に送られるメッセージは、他のすべての因子から変数ノード$\nu_{k}$に届くメッセージの積である。因子の中で、他の変数は全部、周辺化して対象変数だけ残す。

$$
m_{\nu_k \to f}(\nu_k)= \prod_{f' \in F_{\nu_k} \setminus \{f\}} m_{f' \to \nu_k}(\nu_k)
$$

## 具体的な計算例

簡単な例で計算の過程を確認する。ここでは変数ノードとして$X,Y,Z$、因子ノードとして$\psi_X,f,g$があるとする。この因子グラフから確率伝搬を使って$p(Y)$を計算する。離散$X,Y,Z \in \{0,1\}$の例ではあるが、連続なら総和が積分に変わることになる。

グラフ構造として$\psi_X(X)\rightarrow
X \rightarrow f(X,Y) \rightarrow Y \leftarrow g(Y,Z) \leftarrow Z$を仮定する。

$\psi_X(X)$は

$$
\psi_X(X)=
\begin{cases}
9 & (X=0) \\
1 & (X=1)
\end{cases}
$$

$f(X,Y)$は、

$$
\begin{array}{c|cc}
 & Y=0 & Y=1\\\hline
X=0 & 3 & 1\\
X=1 & 1 & 3
\end{array}
$$

$g(Y,Z)$は、

$$
\begin{array}{c|cc}
 & Z=0 & Z=1\\\hline
Y=0 & 2 & 1\\
Y=1 & 1 & 2
\end{array}
$$
である。隣接集合は$F_{X} = \{ \psi_X, f\}, F_{Y} = \{ f, g\}, F_{Z} = \{ g\}$である。

## $p(Y)$の計算

ここから$p(Y)$の周辺分布を求めていく。一般式では、変数ノード$\nu_{k}$における周辺分布の値は、そのノードに流れ込んでくるすべてのメッセージの積を計算すればよいことになっている。$F_{\nu_{k}}$はその変数につながる因子集合で、周辺分布は入ってくるメッセージの積で表される。$Z$は正規化定数。
$$
p({\nu_{k}}) = \frac{1}{Z}\prod_{f \in F_{\nu_{k}}} m_{f \to \nu_{k}}(\nu_{k})
$$

つまり、ここでは$F_{Y} = \{ f, g\}$より

$$
\begin{align}
p(Y = y) 
&= \frac{1}{Z} \prod_{\phi \in F_{Y}} m_{\phi \to Y}(Y) \\
&\propto m_{f \to Y}(y) \cdot m_{g \to Y}(y)
\end{align}
$$

### $m_{f \to Y}(y)$の計算

$m_{f \to Y}(y)$を求める。一般式では、因子ノードから変数ノードへのメッセージは、計算したい変数$\nu_{j}$をのぞくすべての変数について、因子関数の値と他のメッセージの積を積分(和)したものである。そして、因子$f$に送るときは、$f$以外の因子から来た情報を全部掛けることで自分の情報のダブルカウントを避ける。

$$
m_{f \to \nu_{j}}(\nu_{j}) = \sum_{f(\mathbf{V}) \setminus \nu_{j}} f(\mathbf{V}) \prod_{i \neq j}m_{\nu_{i} \to f}(\nu_{i})
$$

つまり、$m_{f \to Y}(y)$の場合、$f(\mathbf{V}) \setminus \nu_{j} = f(x,y) \setminus y = \{x \}$なので、

$$
\begin{align}
m_{f \to Y}(y)
&= \sum_{x} f(x,y) m_{X \to f}(x)
\end{align}
$$

##### $m_{X \to f}(x)$の計算

$m_{X \to f}(x)$を求める必要がある。変数ノード$\nu_{k}$から因子$f$に送られるメッセージは、他のすべての因子から変数ノード$\nu_{k}$に届くメッセージの積である。因子の中で、他の変数は全部、周辺化して対象変数だけ残す。

$$
m_{\nu_k \to f}(\nu_k)= \prod_{f' \in F_{\nu_k} \setminus \{f\}} m_{f' \to \nu_k}(\nu_k)
$$

つまり、一般式の$f$とノード$f$がややこしいが、事前因子なので、$F_{X} = \{\psi_X, f\} \setminus \{ f \} = \{ \psi_X \} $より

$$
\begin{align}
m_{X \to f}(x)
&= \prod_{\phi \in F_{X} \setminus \{ f\}} m_{\phi \to X}(x) \\
&= \prod_{\phi \in \psi_X} m_{\phi \to X}(x) \\
&= m_{\psi_X \to X}(x) \\
&= \psi_{X}(x)
\end{align}
$$

よって、$m_{f \to Y}(y)$は

$$
\begin{align}
m_{f \to Y}(y)
&= \sum_{x} f(x,y) m_{X \to f}(x) \\
&= \sum_{x} f(x,y) \psi_{X}(x) \\
\end{align}
$$

### $m_{g \to Y}(y)$の計算

$m_{g \to Y}(y)$を求める。一般式では、因子ノードから変数ノードへのメッセージは、計算したい変数$\nu_{j}$をのぞくすべての変数について、因子関数の値と他のメッセージの積を積分(和)したものである。そして、因子$f$に送るときは、$f$以外の因子から来た情報を全部掛けることで自分の情報のダブルカウントを避ける(再掲)。

$$
m_{f \to \nu_{j}}(\nu_{j}) = \sum_{f(\mathbf{V}) \setminus \nu_{j}} f(\mathbf{V}) \prod_{i \neq j}m_{\nu_{i} \to f}(\nu_{i})
$$

つまり、$m_{g \to Y}(y)$の場合、$f(\mathbf{V}) \setminus \nu_{j} = f(z,y) \setminus y = \{z \}$なので

$$
\begin{align}
m_{g \to Y}(y)
&= \sum_{z} f(z,y) m_{Z \to g}(z)
\end{align}
$$

##### $m_{Z \to g}(z)$の計算

$m_{Z \to g}(z)$を求める必要がある。変数ノード$\nu_{k}$から因子$f$に送られるメッセージは、他のすべての因子から変数ノード$\nu_{k}$に届くメッセージの積である。因子の中で、他の変数は全部、周辺化して対象変数だけ残す。

$$
m_{\nu_k \to f}(\nu_k)= \prod_{f' \in F_{\nu_k} \setminus \{f\}} m_{f' \to \nu_k}(\nu_k)
$$

つまり、一般式の$f$とノード$f$がややこしいが、$F_{Z} = \{g \} \setminus \{ g \} = \emptyset $より

$$
\begin{align}
m_{Z \to g}(z)
&= \prod_{\phi \in F_{Z} \setminus \{ g\}} m_{\phi \to g}(z) \\
&= \prod_{\phi \in \emptyset} m_{\phi \to g}(z) \\
&= \prod_{\emptyset}(・) \\
&= 1 \\
\end{align}
$$

よって、$m_{g \to Y}(y)$は

$$
\begin{align}
m_{g \to Y}(y)
&= \sum_{z} g(z,y) m_{Z \to g}(z) \\
&= \sum_{z} g(z,y) \cdot 1 \\
\end{align}
$$

## $p(Y)$の計算に戻る

これまでの内容をもとに、$p(Y)$を計算する。$F_{Y} = \{ f, g\}$より

$$
\begin{align}
p(Y = y) 
&= \frac{1}{Z} \prod_{\phi \in F_{Y}} m_{\phi \to Y}(Y) \\
\tilde{p}(Y = y) 
&= m_{f \to Y}(y) \cdot m_{g \to Y}(y) \\
&= \left( \sum_{x} f(x,y) \psi_{X}(x)\right) \cdot \left( \sum_{z} g(z,y)  \cdot 1 \right)\\
\end{align}
$$



$y=0$のとき、

$$
\begin{align}
\tilde{p}(Y = 0) 
&= \left( \sum_{x} f(x,y=0) \psi_{X}(x) \right) \cdot \left( \sum_{z} g(z,y=0)  \cdot 1 \right) \\
&= \left(  f(x=0,y=0) \psi_{X}(x=0) + f(x=1,y=0) \psi_{X}(x=1) \right) \cdot \left( g(z=0,y=0) \cdot 1 + g(z=1,y=0) \cdot 1 \right) \\
&= \left(  3 \cdot 9 + 1 \cdot 1 \right) \cdot \left( 2 \cdot 1 + 1\cdot 1 \right) \\
&= 28\cdot 3 \\
&= 84
\end{align}
$$

$y=1$のとき、

$$
\begin{align}
\tilde{p}(Y = 1) 
&= \left( \sum_{x} f(x,y=1) \psi_{X}(x) \right) \cdot \left( \sum_{z} g(z,y=1)  \cdot 1 \right) \\
&= \left(  f(x=0,y=1) \psi_{X}(x=0) + f(x=1,y=1) \psi_{X}(x=1) \right) \cdot \left( g(z=0,y=1) \cdot 1 + g(z=1,y=1) \cdot 1 \right) \\
&= \left(  1 \cdot 9 + 3 \cdot 1 \right) \cdot \left( 1 \cdot 1 +  2 \cdot 1 \right) \\
&= 12 \cdot 3 \\
&= 36
\end{align}
$$

正規化定数$Z$は

$$
\begin{align}
Z &= 84 + 36 \\
&= 120
\end{align}
$$

ゆえに$p(Y)$の周辺分布は

$$
\begin{align}
p(Y = 0) 
&= \frac{1}{120} \prod_{\phi \in F_{Y}} m_{\phi \to Y}(Y=0) \\
&= \frac{1}{120} \left( \sum_{x} f(x,y=0) \psi_{X}(x) \right) \cdot \left( \sum_{z} g(z,y=0)  \cdot 1 \right) \\
&= \frac{84}{120}\\
&= 0.7\\
\\
p(Y = 1)&= \frac{1}{120} \prod_{\phi \in F_{Y}} m_{\phi \to Y}(Y=1) \\
&= \frac{1}{120} \left( \sum_{x} f(x,y=1) \psi_{X}(x) \right) \cdot \left( \sum_{z} g(z,y=1)  \cdot 1 \right) \\
&= \frac{36}{120}\\
&= 0.3\\
\end{align}
$$
となる。

trueskillの文脈では、変数から因子、因子から変数のメッセージは下記のように表現される。積分が総和になっているが本質は変わらず、因子から変数へは、

$$
\mu_{f \to x}(x) = \sum_{\mathbf{X} \setminus \{x\}} \left( f(\mathbf{X}) \prod_{y \in n(f) \setminus \{x\}} \mu_{y \to f}(y) \right)
$$

変数から因子へは下記のように表現される。ここで$n(x)$は変数$x$の近傍（隣接ノード）を表す。

$$
\mu_{x \to f}(x) = \prod_{h \in n(x) \setminus \{f\}} \mu_{h \to x}(x)
$$

trueskillではメッセージは常に正規分布であり、積・周辺化が解析的に計算できる。下記、雑なメモではあるが1vs1でp1が勝利したときのレーティングの更新過程の計算メモ。

## trueskillレーティングの更新過程の計算メモ
途中`N(m, sigma)`の表記で分散と標準偏差の書き間違いがある・・・

<img src='./elo P1.png'>
<img src='./elo P2.png'>
<img src='./elo P3.png'>

小数点の関係か、どこかで計算ミスしているのか、ライブラリの計算結果と少しズレている。

In [1]:
import trueskill
ts = trueskill.TrueSkill(beta=4.167)
r1 = ts.Rating(mu=25, sigma=8.333)  # 1P's skill
r2 = ts.Rating(mu=25, sigma=8.333)  # 2P's skill
new_r1, new_r2 = trueskill.rate_1vs1(r1, r2)
print(new_r1)
print(new_r2)

trueskill.Rating(mu=29.396, sigma=7.171)
trueskill.Rating(mu=20.604, sigma=7.171)


スクラッチで1on1のtrueskillのレーティング計算を書いたのが下記。

In [2]:
import trueskill_1on1_scratch as ts

game = ts.GameInfo(
    beta=4.1667,
    dynamics_factor=0.083333,
    draw_probability=0.10
)

player_A = ts.Rating(25.0, 8.333)
player_B = ts.Rating(25.0, 8.333)

new_A = ts.calculate_new_rating(game, player_A, player_B, ts.PairwiseComparison.WIN)
print("A updated:", new_A)
new_B = ts.calculate_new_rating(game, player_B, player_A, ts.PairwiseComparison.LOSE)
print("B updated:", new_B)

=== TrueSkill Update Start ===
[1] prior------------------------------------------------------------------
self = Rating(mu=25.000, sigma=8.333)
opp  = Rating(mu=25.000, sigma=8.333)
comparison = PairwiseComparison.WIN

[2] performance dist------------------------------------------------------------------
N(μ1 = 25.00, σ1^2 + β^2 = 86.800)
N(μ2 = 25.00, σ2^2 + β^2 = 86.800)

[3] performance difference------------------------------------------------------------------
μ1-μ2 = 0.000
√(σ1^2 + σ2^2 + 2β^2)= 13.176
σ1^2 + σ2^2 + 2β^2 = 173.601
N(μ1-μ2 = 0.000, σ1^2 + σ2^2 + 2β^2 = 173.601)

[4] truncation correction------------------------------------------------------------------
v(t) = 0.798, w(t) = 0.637
m^*_d = 10.513, v^*_d = 63.083
π_prior = 0.006, π_post = 0.016
τ_prior = 0.000, τ_post = 0.167
m_msg_tmp = 0.010, τ_msg_tmp = 0.167
π_msg = 16.513, τ_msg = 99.091
N(π_msg = 16.513, τ_msg = 99.091)

[5] performance update------------------------------------------------------------------
μ_

A updated: Rating(mu=29.205, sigma=7.194)
B updated: Rating(mu=20.795, sigma=7.194)


## 参考文献
- [Computing Your Skill](https://www.moserware.com/2010/03/computing-your-skill.html)
- [The Math Behind TrueSkill](https://www.moserware.com/assets/computing-your-skill/The%20Math%20Behind%20TrueSkill.pdf)
- [moserware/Skills: A detailed implementation of the TrueSkill algorithm to go along with my "Computing Your Skill" blog post](https://github.com/moserware/Skills)
- [TrueSkillTM: A Bayesian Skill Rating System](https://proceedings.neurips.cc/paper_files/paper/2006/file/f44ee263952e65b3610b8ba51229d1f9-Paper.pdf)
- [TrueSkill — trueskill 0.4.5 documentation](https://trueskill.org/)
- [Andy Jones](https://andrewcharlesjones.github.io/journal/belief-propagation.html)
- [誰でも分かるTrueSkill #機械学習 - Qiita](https://qiita.com/kkjk21/items/aac8e26a2e68c4659c62)